# Legacy Notebook

This notebook is retained for historical reference only.

The main Colab entry point for the project is now `chess_model_run_git.ipynb`, which targets the current reasoning-GRPO fine-tuning path on the refactored branch.


In [1]:
#@title Runtime Parameters
LOCAL_SMOKE_MODE = True  #@param {type:"boolean"}
REPO_URL = "https://github.com/noamdwc/grpo_chess.git"  #@param {type:"string"}
REPO_REF = "feature/lightning_training"  #@param {type:"string"}
LOCAL_REPO_PATH = "~/repos/grpo_chess"  #@param {type:"string"}
USE_DRIVE = True  #@param {type:"boolean"}
DRIVE_ARTIFACT_ROOT = "/content/drive/MyDrive/data/grpo-chess/grpo_9m_direct"  #@param {type:"string"}
LOCAL_ARTIFACT_ROOT = "./artifacts/grpo_9m_direct_smoke"  #@param {type:"string"}
LOCAL_PRECONVERTED_CKPT_PATH = "checkpoints/jax_9m_converted.pt"  #@param {type:"string"}
SKIP_DEP_INSTALL_LOCAL_SMOKE = True  #@param {type:"boolean"}
WANDB_API_KEY = ""  #@param {type:"string"}
RUN_NAME_SUFFIX = ""  #@param {type:"string"}
GRPO_CONFIG_BASE = "grpo_9m_direct.yaml"  #@param {type:"string"}


In [2]:
# Setup workspace and repo
import shutil
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if LOCAL_SMOKE_MODE:
    repo = Path(LOCAL_REPO_PATH).expanduser().resolve()
    if not repo.exists():
        raise FileNotFoundError(f'LOCAL_REPO_PATH does not exist: {repo}')
    %cd {repo}
else:
    if USE_DRIVE and IN_COLAB:
        from google.colab import drive
        drive.mount('/content/drive')

    repo = Path('/content/grpo_chess')
    if repo.exists():
        shutil.rmtree(repo)

    !git clone {REPO_URL} /content/grpo_chess
    %cd /content/grpo_chess
    !git fetch --all --tags
    !git checkout {REPO_REF}
    !git submodule update --init --recursive

repo = Path.cwd().resolve()
if str(repo) not in sys.path:
    sys.path.append(str(repo))

print('Mode:', 'local_smoke' if LOCAL_SMOKE_MODE else 'colab_full')
print('Repo ready at', repo)
print('Python:', sys.version.split()[0])


/Users/noamc/repos/grpo_chess
Mode: local_smoke
Repo ready at /Users/noamc/repos/grpo_chess
Python: 3.14.1


In [3]:
# Enable autoreload for local repo modules (with fallback for older IPython builds)
import importlib
import sys

ip = get_ipython()
try:
    ip.run_line_magic('load_ext', 'autoreload')
    ip.run_line_magic('autoreload', '2')
    print('Autoreload enabled for Python modules.')
except Exception as exc:
    print(f'Autoreload extension unavailable in this runtime: {exc!r}')

    def reload_project_modules(prefixes=('src', 'searchless_chess.src')):
        """Best-effort reload for project modules when autoreload is unavailable."""
        reloaded = []
        for name, module in list(sys.modules.items()):
            if module is None:
                continue
            if not any(name == p or name.startswith(p + '.') for p in prefixes):
                continue
            try:
                importlib.reload(module)
                reloaded.append(name)
            except Exception:
                pass
        print(f'Reloaded {len(reloaded)} modules.')

    print('Use reload_project_modules() after editing files.')

print('Note: module reload does not replace restarts after binary package upgrades.')


Autoreload enabled for Python modules.
Note: module reload does not replace restarts after binary package upgrades.


In [ ]:
# Install dependencies
from pathlib import Path

deps_ready = Path('/tmp/grpo_9m_colab_deps_ready')

if LOCAL_SMOKE_MODE and SKIP_DEP_INSTALL_LOCAL_SMOKE:
    print('Skipping dependency installation in local smoke mode (SKIP_DEP_INSTALL_LOCAL_SMOKE=True).')
    !python -V
    !which stockfish || true
else:
    if not deps_ready.exists():
        %pip install -q --upgrade pip setuptools wheel
        !grep -vE '^numpy==' requirements.txt > /tmp/requirements-colab.txt
        %pip install -q -r /tmp/requirements-colab.txt
        %pip install -q --force-reinstall --no-cache-dir \
            'numpy==2.1.3' 'scipy==1.14.1' 'pandas==2.2.2' 'pyarrow==18.1.0' \
            'requests==2.32.4' 'urllib3<=2.5.0' 'jedi>=0.19.1'
        %pip install -q 'jax==0.4.33' 'jaxlib==0.4.33' 'dm-haiku' 'chex' 'optax' 'orbax-checkpoint' 'grain-nightly' 'absl-py'

        if not LOCAL_SMOKE_MODE:
            !apt-get -qq update
            !apt-get -qq install -y stockfish

        !which stockfish || true
        deps_ready.write_text('ok')
        print('Dependencies installed. Continuing without forced restart.')
        print('If a later import fails, manually restart runtime once and rerun from the top.')
    else:
        !which stockfish || true

try:
    import numpy, scipy, pandas, pyarrow, jax, haiku, orbax.checkpoint, grain.python as pygrain
    print('numpy', numpy.__version__)
    print('scipy', scipy.__version__)
    print('pandas', pandas.__version__)
    print('pyarrow', pyarrow.__version__)
    print('jax', jax.__version__)
    print('haiku', haiku.__version__)
    print('orbax.checkpoint', orbax.checkpoint.__version__)
    print('grain.python', pygrain.__name__)
except Exception as exc:
    print(f'Import check failed: {exc!r}')
    if LOCAL_SMOKE_MODE and SKIP_DEP_INSTALL_LOCAL_SMOKE:
        print('Install missing deps manually or set SKIP_DEP_INSTALL_LOCAL_SMOKE=False and rerun.')
    else:
        raise


Skipping dependency installation in local smoke mode (SKIP_DEP_INSTALL_LOCAL_SMOKE=True).
Python 3.14.1
/opt/homebrew/bin/stockfish


In [ ]:
# W&B auth/config
import os

USE_WANDB = False
wandb_key = WANDB_API_KEY.strip()

if not wandb_key and IN_COLAB:
    try:
        from google.colab import userdata
        wandb_key = (userdata.get('WANDB_API_KEY') or userdata.get('WANDB_KEY') or '').strip()
    except Exception:
        wandb_key = ''

if LOCAL_SMOKE_MODE and not wandb_key:
    USE_WANDB = False
    os.environ['WANDB_DISABLED'] = 'true'
    print('W&B disabled for local smoke mode (no key provided).')
else:
    if not wandb_key:
        raise RuntimeError('W&B key is required for full runs. Set WANDB_API_KEY runtime param or Colab Secret WANDB_API_KEY/WANDB_KEY.')
    os.environ['WANDB_API_KEY'] = wandb_key
    os.environ['WANDB_KEY'] = wandb_key
    print('W&B key configured.')


In [ ]:
# Artifact paths + checkpoint acquisition
from pathlib import Path
import urllib.request
import zipfile

if LOCAL_SMOKE_MODE:
    artifact_root = Path(LOCAL_ARTIFACT_ROOT).expanduser().resolve()
else:
    artifact_root = Path(DRIVE_ARTIFACT_ROOT if USE_DRIVE else '/content/artifacts/grpo_9m_direct')

searchless_ckpt_root = artifact_root / 'searchless_ckpts'
converted_ckpt_root = artifact_root / 'converted_ckpts'
grpo_runs_root = artifact_root / 'grpo_runs'

for d in [artifact_root, searchless_ckpt_root, converted_ckpt_root, grpo_runs_root]:
    d.mkdir(parents=True, exist_ok=True)

if LOCAL_SMOKE_MODE:
    local_ckpt = Path(LOCAL_PRECONVERTED_CKPT_PATH).expanduser()
    if not local_ckpt.is_absolute():
        local_ckpt = (repo / local_ckpt).resolve()
    converted_ckpt_path = local_ckpt
    if not converted_ckpt_path.exists():
        raise FileNotFoundError(
            f'Local smoke mode expects a preconverted checkpoint at {converted_ckpt_path}. '
            'Set LOCAL_PRECONVERTED_CKPT_PATH to an existing .pt file.'
        )
    print('Using local preconverted checkpoint:', converted_ckpt_path)
else:
    nine_m_dir = searchless_ckpt_root / '9M'
    zip_path = searchless_ckpt_root / '9M.zip'
    download_url = 'https://storage.googleapis.com/searchless_chess/checkpoints/9M.zip'

    if not nine_m_dir.exists():
        print('Downloading', download_url)
        urllib.request.urlretrieve(download_url, zip_path)
        print('Extracting', zip_path)
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall(searchless_ckpt_root)
        if zip_path.exists():
            zip_path.unlink()
    else:
        print('Reusing existing 9M checkpoint directory:', nine_m_dir)

    expected_orbax = nine_m_dir / '6400000' / 'params' / 'checkpoint'
    if not expected_orbax.exists():
        raise FileNotFoundError(f'Missing expected Orbax checkpoint file: {expected_orbax}')

    converted_ckpt_path = converted_ckpt_root / 'jax_9m_converted.pt'
    print('Expected Orbax checkpoint found:', expected_orbax)

print('Artifact root:', artifact_root)


In [ ]:
# Convert JAX 9M checkpoint -> PyTorch state dict
if LOCAL_SMOKE_MODE:
    print('Skipping conversion in local smoke mode; using preconverted checkpoint.')
else:
    from src.distill.convert_jax_checkpoint import convert as convert_jax_checkpoint

    if not converted_ckpt_path.exists():
        print('Starting conversion... this can take a while on first run.')
        print('checkpoint_dir:', searchless_ckpt_root)
        print('model_name: 9M')
        print('step: 6400000')
        print('output:', converted_ckpt_path)
        convert_jax_checkpoint(
            checkpoint_dir=str(searchless_ckpt_root),
            model_name='9M',
            step=6400000,
            output=str(converted_ckpt_path),
        )
    else:
        print('Reusing converted checkpoint:', converted_ckpt_path)

if not converted_ckpt_path.exists():
    raise FileNotFoundError(f'Converted checkpoint missing: {converted_ckpt_path}')

print('Converted checkpoint:', converted_ckpt_path)
print('Size (MB):', round(converted_ckpt_path.stat().st_size / (1024 * 1024), 2))


In [ ]:
# Build runtime GRPO config from src/configs/grpo_9m_direct.yaml
from datetime import datetime
import re
import yaml
from src.chess.stockfish import resolve_stockfish_path

base_config_path = repo / 'src' / 'configs' / GRPO_CONFIG_BASE
if not base_config_path.exists():
    raise FileNotFoundError(f'Base GRPO config not found: {base_config_path}')

with base_config_path.open('r') as f:
    cfg = yaml.safe_load(f)

cfg['stockfish']['path'] = resolve_stockfish_path(cfg['stockfish'].get('path'))
cfg['pretrain']['checkpoint_path'] = str(converted_ckpt_path)
cfg['pretrain']['use_9m_direct'] = True
cfg['pretrain']['exact_9m_warmstart'] = True

# Keep baseline eval comparable to checkpoint parity runs.
cfg['eval']['randomize_opening'] = False
cfg['eval']['opening_plies'] = 0
cfg['stockfish']['skill_level'] = 2
cfg['stockfish']['movetime_ms'] = 20

if LOCAL_SMOKE_MODE:
    cfg['training']['num_epochs'] = 1
    cfg['training']['steps_per_epoch'] = 8
    cfg['training']['batch_size'] = min(int(cfg['training'].get('batch_size', 4)), 4)
    cfg['grpo']['num_trajectories'] = 2
    cfg['grpo']['trajectory_depth'] = 4
    cfg['grpo']['eval_every_n_epochs'] = 1
    cfg['eval']['games'] = 2

timestamp = datetime.utcnow().strftime('%Y%m%d-%H%M%S')
suffix = RUN_NAME_SUFFIX.strip()
run_prefix = 'grpo-9m-direct-smoke' if LOCAL_SMOKE_MODE else 'grpo-9m-direct'
if suffix:
    safe_suffix = re.sub(r'[^a-zA-Z0-9_.-]+', '-', suffix)
    run_name = f'{run_prefix}-{timestamp}-{safe_suffix}'
else:
    run_name = f'{run_prefix}-{timestamp}'

run_checkpoint_dir = grpo_runs_root / run_name
run_checkpoint_dir.mkdir(parents=True, exist_ok=True)
cfg['training']['checkpoint_dir'] = str(run_checkpoint_dir)
cfg['training']['use_wandb'] = bool(USE_WANDB)
cfg['training']['wandb_project'] = 'Chess-GRPO-Bot'

runtime_config_name = 'grpo_9m_colab_runtime.yaml'
runtime_config_path = repo / 'src' / 'configs' / runtime_config_name
with runtime_config_path.open('w') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print('Runtime config saved:', runtime_config_path)
print('mode:', 'local_smoke' if LOCAL_SMOKE_MODE else 'colab_full')
print('checkpoint_dir:', cfg['training']['checkpoint_dir'])
print('use_wandb:', cfg['training']['use_wandb'])
print('num_epochs:', cfg['training']['num_epochs'])
print('batch_size:', cfg['training']['batch_size'])
print('steps_per_epoch:', cfg['training']['steps_per_epoch'])
print('num_trajectories:', cfg['grpo']['num_trajectories'])
print('trajectory_depth:', cfg['grpo']['trajectory_depth'])
print('eval.games:', cfg['eval']['games'])
print('eval.randomize_opening:', cfg['eval']['randomize_opening'])
print('eval.opening_plies:', cfg['eval']['opening_plies'])
print('stockfish.path:', cfg['stockfish']['path'])
print('stockfish.skill_level:', cfg['stockfish']['skill_level'])
print('stockfish.movetime_ms:', cfg['stockfish']['movetime_ms'])
print('pretrain.checkpoint_path:', cfg['pretrain']['checkpoint_path'])
print('pretrain.exact_9m_warmstart:', cfg['pretrain']['exact_9m_warmstart'])


In [ ]:
# Launch GRPO training
import torch
from src.train_self_play import train as grpo_train

torch.set_float32_matmul_precision('high')
print('Launching GRPO with config:', runtime_config_name)
grpo_train(
    config_path=runtime_config_name,
    dataloader_kwargs={'num_workers': 0},
)

In [ ]:
# Verify artifacts and surface run pointers
import json

ckpt_files = sorted(run_checkpoint_dir.glob('*.ckpt'), key=lambda p: p.stat().st_mtime)
if not ckpt_files:
    raise RuntimeError(f'No .ckpt files found in {run_checkpoint_dir}')

latest_ckpt = ckpt_files[-1]
print('Checkpoint dir:', run_checkpoint_dir)
print('Num checkpoints:', len(ckpt_files))
print('Latest checkpoint:', latest_ckpt)

run_url = os.environ.get('WANDB_RUN_URL', '').strip()
if run_url:
    print('W&B run URL:', run_url)
else:
    metadata_files = sorted((repo / 'wandb').glob('**/wandb-metadata.json'), key=lambda p: p.stat().st_mtime)
    if metadata_files:
        latest_meta = metadata_files[-1]
        try:
            payload = json.loads(latest_meta.read_text())
            guessed_url = payload.get('url') or payload.get('run_url')
            if guessed_url:
                print('W&B run URL (metadata):', guessed_url)
            else:
                print('Latest W&B metadata file:', latest_meta)
        except Exception:
            print('Latest W&B metadata file:', latest_meta)
    else:
        print('No local W&B metadata found under', repo / 'wandb')

print('Run complete.')

## Troubleshooting

- **Local smoke mode**: set `LOCAL_SMOKE_MODE=True`, `LOCAL_REPO_PATH` to your repo, and `LOCAL_PRECONVERTED_CKPT_PATH` to an existing `jax_9m_converted.pt`.
- **W&B in local smoke**: optional. Leave key empty to disable W&B for smoke runs.
- **JAX conversion import errors**: rerun dependency cell and ensure required deps are installed (disable `SKIP_DEP_INSTALL_LOCAL_SMOKE` if needed).
- **Stockfish path errors**: install Stockfish and ensure it is discoverable on PATH, or set `STOCKFISH_PATH`.
- **Still failing in conversion**: run the conversion cell as-is and copy the full traceback; it shows the exact missing module/file/ABI error.


In [ ]:

import sys
print('Python:', sys.version)
print('Recursion limit:', sys.getrecursionlimit())


In [ ]:

# Simulate the notebook flow
LOCAL_SMOKE_MODE = True
LOCAL_REPO_PATH = "/Users/noamc/repos/grpo_chess"
SKIP_DEP_INSTALL_LOCAL_SMOKE = True
LOCAL_PRECONVERTED_CKPT_PATH = "checkpoints/jax_9m_converted.pt"
LOCAL_ARTIFACT_ROOT = "./artifacts/grpo_9m_direct_smoke"
WANDB_API_KEY = ""
RUN_NAME_SUFFIX = ""
GRPO_CONFIG_BASE = "grpo_9m_direct.yaml"

import shutil
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
repo = Path(LOCAL_REPO_PATH).expanduser().resolve()
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

print('Mode:', 'local_smoke' if LOCAL_SMOKE_MODE else 'colab_full')
print('Repo ready at', repo)
print('IN_COLAB:', IN_COLAB)


In [ ]:

# Enable autoreload (like the notebook does)
ip = get_ipython()
try:
    ip.run_line_magic('load_ext', 'autoreload')
    ip.run_line_magic('autoreload', '2')
    print('Autoreload enabled')
except Exception as exc:
    print(f'Autoreload unavailable: {exc!r}')


In [ ]:

# W&B disabled for local smoke
import os
os.environ['WANDB_DISABLED'] = 'true'

# Setup artifact paths
artifact_root = Path(LOCAL_ARTIFACT_ROOT).expanduser().resolve()
grpo_runs_root = artifact_root / 'grpo_runs'
grpo_runs_root.mkdir(parents=True, exist_ok=True)

converted_ckpt_path = (repo / LOCAL_PRECONVERTED_CKPT_PATH).resolve()
print('Checkpoint:', converted_ckpt_path, 'exists:', converted_ckpt_path.exists())
